# Using SGD with momentum

In [83]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [84]:
df = pd.read_csv("/kaggle/input/datasets/mudarc/my-dataset-2/heart.csv")

In [85]:
df.head(6)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
5,39,M,NAP,120,339,0,Normal,170,N,0.0,Up,0


In [86]:
df.select_dtypes(include='object').nunique()

Sex               2
ChestPainType     4
RestingECG        3
ExerciseAngina    2
ST_Slope          3
dtype: int64

In [87]:
df['Sex'].unique()

array(['M', 'F'], dtype=object)

In [88]:
# lowercasing all the columns names

df.columns = df.columns.str.lower()

In [89]:
# checking for null values
df.isna().sum().sum()

np.int64(0)

In [90]:
df.select_dtypes(include=['float', 'int']).describe()

,age,restingbp,cholesterol,fastingbs,maxhr,oldpeak,heartdisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [91]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

In [92]:
transform = ColumnTransformer(
    transformers=[
        ('t1', MinMaxScaler(), ['fastingbs']),
        ('t2', StandardScaler(), ['age', 'restingbp', 'cholesterol', 'maxhr', 'oldpeak'])
    ], remainder='drop'
)

temp = transform.fit_transform(df)

In [93]:
cols = transform.get_feature_names_out().tolist()
cols = [i.split("__")[-1] for i in cols]
cols

['fastingbs', 'age', 'restingbp', 'cholesterol', 'maxhr', 'oldpeak']

In [94]:
temp = pd.DataFrame(temp, columns=cols)
temp.head()

,fastingbs,age,restingbp,cholesterol,maxhr,oldpeak
0,0.0,-1.433140,0.410909,0.825070,1.382928,-0.832432
1,0.0,-0.478484,1.491752,-0.171961,0.754157,0.105664
2,0.0,-1.751359,-0.129513,0.770188,-1.525138,-0.832432
3,0.0,-0.584556,0.302825,0.139040,-1.132156,0.574711
4,0.0,0.051881,0.951331,-0.034755,-0.581981,-0.832432


In [95]:
df.update(temp)
df.head(6)

,age,sex,chestpaintype,restingbp,cholesterol,fastingbs,restingecg,maxhr,exerciseangina,oldpeak,st_slope,heartdisease
0,-1.433140,M,ATA,0.410909,0.825070,0,Normal,1.382928,N,-0.832432,Up,0
1,-0.478484,F,NAP,1.491752,-0.171961,0,Normal,0.754157,N,0.105664,Flat,1
2,-1.751359,M,ATA,-0.129513,0.770188,0,ST,-1.525138,N,-0.832432,Up,0
3,-0.584556,F,ASY,0.302825,0.139040,0,Normal,-1.132156,Y,0.574711,Flat,1
4,0.051881,M,NAP,0.951331,-0.034755,0,Normal,-0.581981,N,-0.832432,Up,0
5,-1.539213,M,NAP,-0.669935,1.282424,0,Normal,1.304332,N,-0.832432,Up,0


In [96]:
# using one hot encoding

temp = pd.get_dummies(df.select_dtypes(include='object'), dtype=int)

temp.sample(3)

,sex_F,sex_M,chestpaintype_ASY,chestpaintype_ATA,chestpaintype_NAP,chestpaintype_TA,restingecg_LVH,restingecg_Normal,restingecg_ST,exerciseangina_N,exerciseangina_Y,st_slope_Down,st_slope_Flat,st_slope_Up
794,0,1,0,0,1,0,0,1,0,1,0,0,0,1
586,0,1,1,0,0,0,0,1,0,0,1,0,1,0
778,0,1,1,0,0,0,1,0,0,0,1,0,1,0


In [97]:
data = df.drop(columns=(df.select_dtypes(include='object')))
data = pd.concat([data, temp], axis=1)
data.sample(3)

,age,restingbp,cholesterol,fastingbs,maxhr,oldpeak,heartdisease,sex_F,sex_M,chestpaintype_ASY,...,chestpaintype_NAP,chestpaintype_TA,restingecg_LVH,restingecg_Normal,restingecg_ST,exerciseangina_N,exerciseangina_Y,st_slope_Down,st_slope_Flat,st_slope_Up
515,1.006537,-0.129513,-1.818435,1,0.911350,1.981855,0,0,1,0,...,1,0,0,0,1,1,0,0,1,0
190,-0.796702,2.572596,0.742747,0,-0.660578,-0.832432,0,0,1,1,...,0,0,0,0,1,1,0,0,0,1
596,0.370100,-0.561850,0.596393,0,-1.446542,-0.832432,1,0,1,1,...,0,0,1,0,0,1,0,0,1,0


In [98]:
from sklearn.model_selection import train_test_split as ttt

In [99]:
x = data.drop(columns=['heartdisease'])
y = data[['heartdisease']]

x_train, x_test, y_train, y_test = ttt(x, y, test_size=0.2, random_state=42)

In [100]:
# using SGD with momentum
from sklearn.neural_network import MLPClassifier       # used for SGD with momentum
from sklearn.linear_model import SGDClassifier

In [101]:
model = MLPClassifier(
    hidden_layer_sizes=(),       # Empty tuple = no hidden layers (behaves like Logistic Regression)
    solver='sgd',                # Activates Stochastic Gradient Descent
    momentum=0.9,                # Enables momentum acceleration
    learning_rate_init=0.005,    # Slightly lowered learning rate to prevent momentum from overshooting
    nesterovs_momentum=True,     # Crucial: Acts as an anticipatory "brake" to stabilize momentum
    max_iter=1000,
    shuffle=True,
    early_stopping=True,         # Keeps validation monitoring on
    n_iter_no_change=15,         # Increased patience from 6 to 15 so momentum has time to settle down
    random_state=42
)

model.fit(x_train, y_train)
prediction = model.predict(x_test)


In [102]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print(f"ACCURACY SCORE : \n{accuracy_score(y_test, prediction)}\n")
print(f"PRECISION SCORE : \n{precision_score(y_test, prediction, average='weighted')}\n")
print(f"RECALL SCORE : \n{recall_score(y_test, prediction, average='weighted')}\n")
print(f"CONFUSION MATRIX : \n{confusion_matrix(y_test, prediction)}")

ACCURACY SCORE : 
0.8043478260869565

PRECISION SCORE : 
0.8114319129095245

RECALL SCORE : 
0.8043478260869565

CONFUSION MATRIX : 
[[64 13]
 [23 84]]


In [103]:
model2 = SGDClassifier(
    loss='log_loss',         # <-- Important: Added to keep it as Logistic Regression
    penalty='l1',            # <-- Fixed: Added quotes around 'l1'
    alpha=0.01,
    max_iter=1000,
    shuffle=True,
    early_stopping=True,
    n_iter_no_change=6,
    random_state=42
)

model2.fit(x_train, y_train)
prediction2 = model2.predict(x_test)


In [104]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print(f"ACCURACY SCORE : \n{accuracy_score(y_test, prediction2)}\n")
print(f"PRECISION SCORE : \n{precision_score(y_test, prediction2, average='weighted')}\n")
print(f"RECALL SCORE : \n{recall_score(y_test, prediction2, average='weighted')}\n")
print(f"CONFUSION MATRIX : \n{confusion_matrix(y_test, prediction2)}")

ACCURACY SCORE : 
0.8532608695652174

PRECISION SCORE : 
0.8590064691194981

RECALL SCORE : 
0.8532608695652174

CONFUSION MATRIX : 
[[68  9]
 [18 89]]
